<a href="https://colab.research.google.com/github/haskinse/bee2041_empirical_project/blob/main/source_code/02_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [57]:
from google.colab import drive # connect to google drive
drive.mount("/content/drive")

project_path = "/content/drive/MyDrive/bee2041_empirical_project" # define project path

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [58]:
import pandas as pd # library for data manipulation and tables

In [60]:
def clean_wiki_table(table): # a function to clean the wikipedia tables into structured data frames

  table.columns = table.iloc[0]  # set first row as column names
  table = table.reset_index(drop=True) # reset index cleanly

  table = table.replace(r"\[.*?\]", "", regex = True) # remove footnotes like [1] or [a]

  # extract the release date from the album details column text
  table["release_date"] = table["Album details"].str.extract(r"Released:\s*([A-Za-z]+\s+\d{1,2},\s+\d{4})")

  table["release_date"] = pd.to_datetime(table['release_date']) # convert the date from string to datetime format for easier sorting

  table["label"] = table["Album details"].str.extract(r"Label:\s*(.*?)\s*Formats:") # extract the label from the details column and store in a new column

  table["us_sales"] = table["Sales[a]"].str.extract(r"US:\s*([\d,]+)") # extract the us sales
  table["uk_sales"] = table["Sales[a]"].str.extract(r"UK:\s*([\d,]+)") # extract the uk sales

  # for the sales data, remove commas, convert to numeric and fill missing values with zero
  table[["uk_sales", "us_sales"]] = (table[["uk_sales", "us_sales"]].replace(",", "", regex=True).apply(pd.to_numeric, errors="coerce").fillna(0).astype(int))

  table["riaa"] = table["Certifications"].str.extract(r"RIAA:\s*(.*?)(?:ARIA:|BPI:|$)") # extract the riaa certification text
  riaa_mult = pd.to_numeric(table["riaa"].str.extract(r"(\d+)")[0], errors="coerce").fillna(1).astype(int) # extract the riaa mulripliers like 2x

  # convert the riaa certification levels into estimated units
  table["riaa_units"] = (table["riaa"].str.contains("Gold", na = False) * 500_000 + table['riaa'].str.contains("Platinum", na=False) * 1_000_000) * riaa_mult

  # extract bpi certification text
  table["bpi"] = table["Certifications"].str.extract(r"BPI:\s*(.*?)(?:RIAA:|ARIA:|MC:|RMNZ:|IFPI DEN:|IFPI NOR:|GLF:|$)")

  bpi_mult = pd.to_numeric(table["bpi"].str.extract(r"(\d+)")[0], errors="coerce").fillna(1).astype(int) # extract bpi multipliers

  # convert the bpi certification levels into estimated units
  table["bpi_units"] = (table["bpi"].str.contains("Gold", na = False) * 100_000 + table["bpi"].str.contains("Platinum", na=False) * 300_000) * bpi_mult

  # remove columns that won't be used any further
  table = table.drop(columns = ["Album details", "Certifications", "AUS [6]", "CAN [8]", "DEN [16]", "IRE [17]", "NZ [9]", "NOR [18]", "SPA [13]", "SWE [14]", "Sales[a]"])

  table = table.rename(columns = {"Title": "album_name", "US [19]": "us_peak", "UK [10]": "uk_peak"}) # standardise column names
  table = table.dropna(subset=["release_date"]).reset_index(drop=True) # drop non-album rows where there is no release date
  table = table.dropna(axis = 1, how = "all") # drop empty columns

  return table # return the cleaned data frame

In [61]:
wiki_studio_albums = pd.read_csv(f"{project_path}/data/raw/wiki_studio_albums.csv") # read raw csv data into a data frame
wiki_studio_albums = clean_wiki_table(wiki_studio_albums) # clean the albums table using the function

wiki_studio_albums.head() # show the first five rows

,album_name,us_peak,uk_peak,release_date,label,us_sales,uk_sales,riaa,riaa_units,bpi,bpi_units
0,Taylor Swift,5,81,2006-10-24,Big Machine,5871000,251320,8× Platinum,8000000,Platinum,300000
1,Fearless,1,5,2008-11-11,Big Machine,7286000,718518,11× Platinum (Diamond),11000000,2× Platinum,600000
2,Speak Now,1,6,2010-10-25,Big Machine,4817000,416676,6× Platinum,6000000,Platinum,300000
3,Red,1,1,2012-10-22,Big Machine,4582000,874532,8× Platinum,8000000,3× Platinum,900000
4,1989,1,1,2014-10-27,Big Machine,6472000,1792380,14× Platinum (Diamond),14000000,6× Platinum,1800000


In [62]:
wiki_tv_albums = pd.read_csv(f"{project_path}/data/raw/wiki_tv_albums.csv") # read raw csv data into a data frame
wiki_tv_albums = clean_wiki_table(wiki_tv_albums) # clean the albums table using the function

wiki_tv_albums.head() # show the first five rows

,album_name,us_peak,uk_peak,release_date,label,us_sales,uk_sales,riaa,riaa_units,bpi,bpi_units
0,Fearless (Taylor's Version),1,1,2021-04-09,Republic,737000,217947,4× Platinum,4000000,Platinum,300000
1,Red (Taylor's Version),1,1,2021-11-12,Republic,950000,357565,6× Platinum,6000000,Platinum,300000
2,Speak Now (Taylor's Version),1,1,2023-07-07,Republic,908000,209302,4× Platinum,4000000,Platinum,300000
3,1989 (Taylor's Version),1,1,2023-10-27,Republic,2389000,390226,4× Platinum,4000000,2× Platinum,600000


In [63]:
album_data = pd.concat([wiki_studio_albums, wiki_tv_albums], ignore_index = True) # combine studio_albums and rerecorded_albums
album_data = album_data.sort_values("release_date") # sort by release date
album_data = album_data.reset_index(drop = True) # reset the index
album_data["album_id"] = album_data.index + 1 # create the album id column to uniquely identify each album

album_data.to_csv(f"{project_path}/data/clean/album_data.csv", index = False) # save clean data to drive

album_data.head() # show first five rows

,album_name,us_peak,uk_peak,release_date,label,us_sales,uk_sales,riaa,riaa_units,bpi,bpi_units,album_id
0,Taylor Swift,5,81,2006-10-24,Big Machine,5871000,251320,8× Platinum,8000000,Platinum,300000,1
1,Fearless,1,5,2008-11-11,Big Machine,7286000,718518,11× Platinum (Diamond),11000000,2× Platinum,600000,2
2,Speak Now,1,6,2010-10-25,Big Machine,4817000,416676,6× Platinum,6000000,Platinum,300000,3
3,Red,1,1,2012-10-22,Big Machine,4582000,874532,8× Platinum,8000000,3× Platinum,900000,4
4,1989,1,1,2014-10-27,Big Machine,6472000,1792380,14× Platinum (Diamond),14000000,6× Platinum,1800000,5
